In [1]:
import pandas as pd
import mysql.connector

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Mukesh@123"
)

print("MySQL connected successfully.")

MySQL connected successfully.


In [4]:
cursor = connection.cursor()

cursor.execute(
    "CREATE DATABASE IF NOT EXISTS packaging_control_tower"
)

print("Database created successfully.")

Database created successfully.


In [5]:
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Mukesh@123",
    database="packaging_control_tower"
)

cursor = connection.cursor()

print("Connected to packaging_control_tower.")

Connected to packaging_control_tower.


In [6]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS machine_operations (
    id INT AUTO_INCREMENT PRIMARY KEY,
    equipment_ID VARCHAR(50),
    interval_start DATETIME,
    production FLOAT,
    downtime FLOAT,
    idle FLOAT,
    performance_loss FLOAT,
    scheduled_downtime FLOAT,
    changes_count FLOAT,
    health_score FLOAT
)
""")

connection.commit()

print("machine_operations table created.")

machine_operations table created.


In [7]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS machine_anomalies (
    id INT AUTO_INCREMENT PRIMARY KEY,
    equipment_ID VARCHAR(50),
    interval_start DATETIME,
    anomaly_prediction INT,
    anomaly_score FLOAT,
    is_anomaly BOOLEAN
)
""")

connection.commit()

print("machine_anomalies table created.")

machine_anomalies table created.


In [8]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS machine_risk (
    id INT AUTO_INCREMENT PRIMARY KEY,
    equipment_ID VARCHAR(50),
    interval_start DATETIME,
    future_risk INT,
    risk_prediction INT,
    risk_probability FLOAT,
    risk_level VARCHAR(20)
)
""")

connection.commit()

print("machine_risk table created.")

machine_risk table created.


In [9]:
cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table[0])

machine_anomalies
machine_operations
machine_risk


In [10]:
features_df = pd.read_csv(
    "../data/processed/features.csv"
)

features_df["interval_start"] = pd.to_datetime(
    features_df["interval_start"],
    errors="coerce"
)

print("Features loaded:")
print(features_df.shape)

Features loaded:
(23376, 170)


In [11]:
machine_df = features_df[
    [
        "equipment_ID",
        "interval_start",
        "%production",
        "%downtime",
        "%idle",
        "%performance_loss",
        "%scheduled_downtime",
        "#changes",
        "health_score"
    ]
].copy()

In [12]:
machine_df = machine_df.rename(
    columns={
        "%production": "production",
        "%downtime": "downtime",
        "%idle": "idle",
        "%performance_loss": "performance_loss",
        "%scheduled_downtime": "scheduled_downtime",
        "#changes": "changes_count"
    }
)

machine_df.head()

,equipment_ID,interval_start,production,downtime,idle,performance_loss,scheduled_downtime,changes_count,health_score
0,s_1,2020-01-01 14:00:00,0.861729,0.052261,0.021523,0.020092,0.044395,18.0,99.906124
1,s_1,2020-01-01 15:00:00,0.870897,0.117704,0.000000,0.011399,0.000000,5.0,99.870897
2,s_1,2020-01-01 17:00:00,0.978483,0.021517,0.000000,0.000000,0.000000,2.0,99.978483
3,s_1,2020-01-01 18:00:00,0.471066,0.468384,0.000000,0.000000,0.060550,8.0,99.531616
4,s_1,2020-01-01 19:00:00,0.987024,0.007778,0.000000,0.005198,0.000000,3.0,99.987024


In [13]:
insert_query = """
INSERT INTO machine_operations
(
    equipment_ID,
    interval_start,
    production,
    downtime,
    idle,
    performance_loss,
    scheduled_downtime,
    changes_count,
    health_score
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

In [14]:
for _, row in machine_df.iterrows():

    cursor.execute(
        insert_query,
        (
            row["equipment_ID"],
            row["interval_start"],
            row["production"],
            row["downtime"],
            row["idle"],
            row["performance_loss"],
            row["scheduled_downtime"],
            row["changes_count"],
            row["health_score"]
        )
    )

connection.commit()

print("Machine operation data inserted.")

Machine operation data inserted.


In [15]:
anomaly_df = pd.read_csv(
    "../data/processed/anomaly_results.csv"
)

anomaly_df["interval_start"] = pd.to_datetime(
    anomaly_df["interval_start"],
    errors="coerce"
)

print(anomaly_df.shape)

(23376, 174)


In [16]:
anomaly_df = anomaly_df[
    [
        "equipment_ID",
        "interval_start",
        "anomaly_prediction",
        "anomaly_score",
        "is_anomaly"
    ]
].copy()

anomaly_df.head()

,equipment_ID,interval_start,anomaly_prediction,anomaly_score,is_anomaly
0,s_1,2020-01-01 14:00:00,1,0.157169,False
1,s_1,2020-01-01 15:00:00,1,0.212733,False
2,s_1,2020-01-01 17:00:00,1,0.197203,False
3,s_1,2020-01-01 18:00:00,1,0.113742,False
4,s_1,2020-01-01 19:00:00,1,0.197903,False


In [17]:
insert_anomaly = """
INSERT INTO machine_anomalies
(
    equipment_ID,
    interval_start,
    anomaly_prediction,
    anomaly_score,
    is_anomaly
)
VALUES (%s, %s, %s, %s, %s)
"""

In [18]:
for _, row in anomaly_df.iterrows():

    cursor.execute(
        insert_anomaly,
        (
            row["equipment_ID"],
            row["interval_start"],
            row["anomaly_prediction"],
            row["anomaly_score"],
            bool(row["is_anomaly"])
        )
    )

connection.commit()

print("Anomaly data inserted.")

Anomaly data inserted.


In [19]:
risk_df = pd.read_csv(
    "../data/processed/risk_results.csv"
)

risk_df["interval_start"] = pd.to_datetime(
    risk_df["interval_start"],
    errors="coerce"
)

print(risk_df.shape)

(4670, 177)


In [20]:
risk_df = risk_df[
    [
        "equipment_ID",
        "interval_start",
        "future_risk",
        "risk_prediction",
        "risk_probability",
        "risk_level"
    ]
].copy()

risk_df.head()

,equipment_ID,interval_start,future_risk,risk_prediction,risk_probability,risk_level
0,s_2,2021-09-29 10:00:00,0,1,61.150925,HIGH
1,s_5,2021-09-29 10:00:00,0,0,48.113310,MEDIUM
2,s_5,2021-09-29 11:00:00,0,0,45.625343,MEDIUM
3,s_3,2021-09-29 11:00:00,0,0,45.030220,MEDIUM
4,s_3,2021-09-29 12:00:00,0,1,50.587630,MEDIUM


In [21]:
insert_risk = """
INSERT INTO machine_risk
(
    equipment_ID,
    interval_start,
    future_risk,
    risk_prediction,
    risk_probability,
    risk_level
)
VALUES (%s, %s, %s, %s, %s, %s)
"""

In [22]:
for _, row in risk_df.iterrows():

    cursor.execute(
        insert_risk,
        (
            row["equipment_ID"],
            row["interval_start"],
            row["future_risk"],
            row["risk_prediction"],
            row["risk_probability"],
            row["risk_level"]
        )
    )

connection.commit()

print("Risk data inserted.")

Risk data inserted.


In [23]:
cursor.execute(
    "SELECT COUNT(*) FROM machine_operations"
)

result = cursor.fetchone()

print("Total machine operation records:", result[0])

Total machine operation records: 23376


In [24]:
cursor.execute("""
SELECT
    equipment_ID,
    COUNT(*) AS records
FROM machine_operations
GROUP BY equipment_ID
ORDER BY equipment_ID
""")

for row in cursor.fetchall():
    print(row)

('s_1', 8973)
('s_2', 2725)
('s_3', 2976)
('s_4', 3959)
('s_5', 4743)


In [25]:
cursor.execute("""
SELECT
    equipment_ID,
    AVG(downtime) AS average_downtime
FROM machine_operations
GROUP BY equipment_ID
ORDER BY average_downtime DESC
""")

for row in cursor.fetchall():
    print(row)

('s_2', 0.19112273332820948)
('s_1', 0.15951517560879966)
('s_3', 0.14075933797986542)
('s_4', 0.0859588789270782)
('s_5', 0.08267045548628639)


In [26]:
cursor.execute("""
SELECT
    equipment_ID,
    COUNT(*) AS high_risk_records,
    AVG(risk_probability) AS average_risk
FROM machine_risk
WHERE risk_prediction = 1
GROUP BY equipment_ID
ORDER BY average_risk DESC
""")

for row in cursor.fetchall():
    print(row)

('s_2', 306, 58.2607834136564)
('s_4', 670, 56.84791559247828)
('s_3', 112, 55.49351011003767)
('s_5', 214, 55.19279836494232)
('s_1', 378, 54.879649227889125)
